# Module 04 — fOU Convergence Probability and DTE Signal

The strategy no longer uses a universal Z-score entry threshold or a fixed multiple of half-life for option maturity.

For the current spread state, Module 04 estimates

\[
P(\tau_\mu \le T),
\]

where \(\tau_\mu\) is the first time the fractional OU spread reaches its fitted equilibrium \(\mu\).

The candidate DTE is the **smallest horizon** satisfying

\[
P(\tau_\mu \le T)\ge 70\%.
\]

If the target is not reached within the permitted search horizon, no statistical trade candidate is generated.

Because fOU is non-Markovian, simulations condition future fractional Gaussian noise on a finite recent history of inferred innovations.

In [ ]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from convergence_signal import (
    calculate_convergence_signal,
    trading_days_to_calendar_days,
)

## 1. Load fitted fOU parameters and spread histories

For development checks you may use the training spread. During the final out-of-sample backtest, the function must be called sequentially using only data available through each date.

In [ ]:
PARAM_FILE = PROJECT_ROOT / "data" / "processed" / "fractional_ou_parameters.parquet"
SPREADS_FILE = PROJECT_ROOT / "data" / "processed" / "training_spreads.pkl"

fou_parameters = pd.read_parquet(PARAM_FILE)

with open(SPREADS_FILE, "rb") as f:
    spreads = pickle.load(f)

fou_parameters.head()

## 2. Signal parameters

`TARGET_PROBABILITY = 0.70` is the agreed statistical confidence target.

`MEMORY_WINDOW` determines how many recent inferred fGn innovations are used to condition the future fractional-noise distribution.

The maximum horizon only caps the search; it does not determine DTE.

In [ ]:
TARGET_PROBABILITY = 0.70
MEMORY_WINDOW = 60
N_PATHS = 5000

# None => search up to 3 drift half-lives.
MAX_HORIZON_DAYS = None

## 3. Inspect one pair

In [ ]:
row = fou_parameters.iloc[0]
pair = row["pair"]
dep = row["dependent"]
indep = row["independent"]

history = spreads[(dep, indep)]

signal, probability_curve = calculate_convergence_signal(
    spread_history=history,
    mu=row["mu"],
    kappa=row["kappa"],
    sigma=row["sigma"],
    hurst=row["hurst"],
    stationary_variance=row["variance"],
    target_probability=TARGET_PROBABILITY,
    memory_window=MEMORY_WINDOW,
    max_horizon_days=MAX_HORIZON_DAYS,
    n_paths=N_PATHS,
)

signal

In [ ]:
if signal.selected_dte_trading_days is not None:
    calendar_dte = trading_days_to_calendar_days(signal.selected_dte_trading_days)

    print(f"Pair: {pair}")
    print(f"Current standardized spread: {signal.current_z:.3f}")
    print(f"Target probability: {signal.target_probability:.1%}")
    print(f"Selected trading-day DTE: {signal.selected_dte_trading_days}")
    print(f"Approx. calendar DTE: {calendar_dte}")
    print(f"Probability at selected DTE: {signal.probability_at_selected_dte:.3f}")
else:
    print(f"{pair}: target probability was not reached. No statistical entry candidate.")

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(probability_curve.index, probability_curve.values)
plt.axhline(TARGET_PROBABILITY, linestyle="--", label="Target probability")

if signal.selected_dte_trading_days is not None:
    plt.axvline(signal.selected_dte_trading_days, linestyle=":", label="Selected DTE")

plt.xlabel("Trading days")
plt.ylabel("P(convergence by T)")
plt.title(f"fOU First-Passage Probability — {pair}")
plt.legend()
plt.show()

## 4. Generate a current statistical signal for every pair

This cell evaluates the most recent observation in each supplied spread history. The final backtester will apply exactly the same logic sequentially.

In [ ]:
signal_rows = []

for _, row in fou_parameters.iterrows():
    pair = row["pair"]
    dep = row["dependent"]
    indep = row["independent"]
    key = (dep, indep)

    if key not in spreads:
        continue

    try:
        sig, _ = calculate_convergence_signal(
            spread_history=spreads[key],
            mu=row["mu"],
            kappa=row["kappa"],
            sigma=row["sigma"],
            hurst=row["hurst"],
            stationary_variance=row["variance"],
            target_probability=TARGET_PROBABILITY,
            memory_window=MEMORY_WINDOW,
            max_horizon_days=MAX_HORIZON_DAYS,
            n_paths=N_PATHS,
            seed=42,
        )
    except Exception as exc:
        print(f"Skipped {pair}: {exc}")
        continue

    record = {
        "pair": pair,
        "dependent": dep,
        "independent": indep,
        **sig.to_dict(),
    }

    record["selected_dte_calendar_days"] = (
        trading_days_to_calendar_days(sig.selected_dte_trading_days)
        if sig.selected_dte_trading_days is not None
        else np.nan
    )

    signal_rows.append(record)

current_signals = pd.DataFrame(signal_rows)
current_signals

## 5. Current candidates

Direction convention:

- `direction = +1`: spread is above equilibrium; later option construction positions for convergence downward.
- `direction = -1`: spread is below equilibrium; later option construction positions for convergence upward.

Module 04 only answers the statistical convergence question. The later options module will decide whether the expected move is economically large enough relative to option premium.

In [ ]:
candidates = current_signals[current_signals["statistical_signal"]].copy()

candidates = candidates.sort_values(
    ["probability_at_max_horizon", "selected_dte_trading_days"],
    ascending=[False, True],
)

candidates

## 6. Save current statistical signal output

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "fou_current_signals.parquet"
current_signals.to_parquet(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")